In [2]:
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)
from IPython.core.display import display, HTML
import scipy.constants as sc
from scipy.optimize import fsolve
from ipywidgets import interact, widgets
%matplotlib notebook

import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as sc
plt.rcParams.update({
    "text.usetex": True,
    'text.latex.preamble': r"\usepackage{amsmath} \boldmath"
})
import os

print(os.getpid())
%cd ../../../

19832
/home/ivan/Documents/Postdoc-PPPL/progs/c++/QPlasma


In [3]:
# --- choose a case to simulate ---
import QSP.examples.XW_1D.xw_1d as xw
xw.mix.reload_module(xw)

case_to_model = "tiny-n3"
# case_to_model = "tiny-n4"
# case_to_model = "tiny-n5"
# case_to_model = "small"
# case_to_model = "ref"

dd = xw.init(case_to_model)

# get some data:
path_root = dd['path-qc']
x_plot = dd['x-plot']
int_states = [i for i in range(len(x_plot))]
x0 = dd["x-source"]
nq = dd["nqx"]
nx = dd["nx"]
coef_coupling = dd["coef-coupling"]

eta = dd['norm-of-h']

Simulation of a --- TINY --- case

---------------------------------------------------------------------
--- Classical parameters ---
nqx, nx = 3, 8
t, nt = 100.000, 400
norm. time step: 2.513e-01
Courant number is: 5.221e-02

---------------------------------------------------------------------
--- QC parameters ---
QC time step: 1.196
QC normalization of a Hamiltonian matrix: 4.758
QC 1/norm_of_H: 2.102e-01
QC source init. value: 7.071e-01
QC time to simulate: 475.824
QC number of time steps: 398


In [4]:
xw.mix.reload_module(xw)
def get_source_position(x_plot, x0):
    id_Q_1, _ = xw.mix.find(x_plot, x0)
    id_Q_2 = id_Q_1 - 1
    return id_Q_1, id_Q_2

In [5]:
# ---------------------------------------------
# --- Profiles for sqrt(OH)
# ---------------------------------------------
xw.mix.reload_module(xw)

wr = {}
wr["B"] = np.sqrt(dd['B-norm']/eta)
wr["n"] = np.sqrt(np.sqrt(dd['n-norm'])/eta)
wr["g"]     = np.sqrt(dd["1/2h"]/eta)
wr["beta"]  = np.sqrt(coef_coupling/eta)
wr["wa"]    = np.sqrt(dd["w-ref-norm"]/eta)
coef_h  = 1. / np.sqrt(2)
coef_h2 = 1. / 2

g = wr["g"]
print("sqrt(1/(2*h*eta)) = {:0.3e}".format(g))

beta = wr["beta"]
print("sqrt(beta/eta) = {:0.3e}".format(beta))

omega_a = wr["wa"]
print("sqrt(omega_a/eta) = {:0.3e}".format(omega_a))

# --- angles for derivatives and source in sqrt(OH) ---
print("")
print("par(beta) = {:0.3f}".format( 2*np.arccos(beta/coef_h) ))
print("par(wa) = {:0.3f}".format( 2*np.arccos(omega_a/coef_h) ))

# --- positions of source ---
id_Q_1, id_Q_2 = get_source_position(x_plot, x0)

print(f"qubit states of the source: {id_Q_1}, {id_Q_2}")
print(" ")
print("3pi = {:0.10f}".format(3*np.pi))
print("2pi = {:0.10f}".format(2*np.pi))
print("pi = {:0.10f}".format(np.pi))
print("---------------------- SQRT PROFILES --------------------")
print("{:>4s} {:>10s}  {:>15s} {:>16s}".format("j", "x", "sqrt(B/eta)", "sqrt(sqrt(n)/eta)"))
for idx in range(len(x_plot)):
    print("{:4d} {:10.3f} {:16.3e} {:16.3e}".format(idx, x_plot[idx], wr["B"][idx], wr["n"][idx]))

# --- write profiles into the file ---
# these profiles are passed to condR with additional coefficients to eliminate
# additional multipliers from Hadamard gates
prof_n = wr["n"]; prof_b = wr["B"]

# - profiles for rows VX and VY -
prof_nvxy = np.array(prof_n)/coef_h
xw.mix.write_profile_condR("sqrt_nvxy", prof_nvxy, path_root)

prof_bvx = np.array(prof_b)/coef_h;
xw.mix.write_profile_condR("sqrt_bvx", prof_bvx, path_root)

prof_bvy = np.array(prof_b); prof_bvy[1:nx-1] = prof_bvy[1:nx-1]/coef_h
xw.mix.write_profile_condR("sqrt_bvy", prof_bvy, path_root)

# - profile for row EX -
prof_nex = np.array(prof_n);
xw.mix.write_profile_condR("sqrt_nex", prof_nex, path_root)

# - profiles for row EY -
prof_ney = np.array(prof_n)/coef_h;
xw.mix.write_profile_condR("sqrt_ney", prof_ney, path_root)

prof_gey = np.array([g/coef_h2]*nx)
prof_gey[1] = g/coef_h;  prof_gey[nx-2] = g/coef_h
xw.mix.write_profile_condR("sqrt_gey", prof_gey, path_root)

# - profiles for row Bz -
prof_gbz = np.array([g/coef_h]*nx)
prof_gbz[1] = g;  prof_gbz[nx-2] = g
prof_gbz[id_Q_1] = g/coef_h2;  prof_gbz[id_Q_2] = g/coef_h2
xw.mix.write_profile_condR("sqrt_gbz", prof_gbz, path_root)

sqrt(1/(2*h*eta)) = 1.478e-01
sqrt(beta/eta) = 1.450e-01
sqrt(omega_a/eta) = 2.838e-01

par(beta) = 2.729
par(wa) = 2.316
qubit states of the source: 4, 3
 
3pi = 9.4247779608
2pi = 6.2831853072
pi = 3.1415926536
---------------------- SQRT PROFILES --------------------
   j          x      sqrt(B/eta) sqrt(sqrt(n)/eta)
   0     -1.000        1.218e-01        4.583e-01
   1     -0.714        1.216e-01        3.615e-01
   2     -0.429        1.856e-01        1.712e-01
   3     -0.143        2.508e-01        4.867e-02
   4      0.143        2.987e-01        2.383e-02
   5      0.429        3.201e-01        9.197e-02
   6      0.714        3.125e-01        1.898e-01
   7      1.000        3.056e-01        2.086e-01
writing a profile of sqrt_nvxy
writing a profile of sqrt_bvx
writing a profile of sqrt_bvy
writing a profile of sqrt_nex
writing a profile of sqrt_ney
writing a profile of sqrt_gey
writing a profile of sqrt_gbz


In [6]:
# ---------------------------------------------
# --- Elements of a Hamiltonian
# ---------------------------------------------
xw.mix.reload_module(xw)

wr = {}
wr["B"] = dd['B-norm']/eta
wr["n"] = np.sqrt(dd['n-norm'])/eta
wr["g"]     = dd["1/2h"]/eta
wr["beta"]  = coef_coupling/eta
wr["wa"]    = dd["w-ref-norm"]/eta
coef_h  = 1. / np.sqrt(2)
coef_h2 = 1. / 2

g = wr["g"]
print("1/(2*h*eta) = {:0.3e}".format(g))

beta = wr["beta"]
print("beta/eta = {:0.3e}".format(beta))

omega_a = wr["wa"]
print("omega_a/eta = {:0.3e}".format(omega_a))
print("---------------------- PROFILES (without sqrt) --------------------")
print("{:>4s} {:>10s}  {:>15s} {:>16s}".format("j", "x", "B/eta", "sqrt(n)/eta"))
for idx in range(len(x_plot)):
    print("{:4d} {:10.3f} {:16.3e} {:16.3e}".format(idx, x_plot[idx], wr["B"][idx], wr["n"][idx]))

1/(2*h*eta) = 2.184e-02
beta/eta = 2.102e-02
omega_a/eta = 8.056e-02
---------------------- PROFILES (without sqrt) --------------------
   j          x            B/eta      sqrt(n)/eta
   0     -1.000        1.483e-02        2.100e-01
   1     -0.714        1.478e-02        1.307e-01
   2     -0.429        3.444e-02        2.931e-02
   3     -0.143        6.289e-02        2.369e-03
   4      0.143        8.921e-02        5.679e-04
   5      0.429        1.025e-01        8.458e-03
   6      0.714        9.768e-02        3.601e-02
   7      1.000        9.337e-02        4.350e-02


In [7]:
# ---------------------------------------------------------------
# --- Initial state ---
# ---------------------------------------------------------------
xw.mix.reload_module(xw)

# size of an initial vector
N_vector = 2**(nq + 3) # 3 is because of the d-register

# source position
id_Q_1, id_Q_2 = get_source_position(x_plot, x0)

# normalization factor of a Hamiltonian (to make its norm be <= 1):
# defined as H_norm = norm_coef * H
norm_coef = 1. / eta

# set 1/sqrt(2) at source positions:
coef_sqrt = 1./np.sqrt(2) # since we have two source points
xr = np.zeros(N_vector)
xi = np.zeros(N_vector)

id_shift_regQ = 2**nq * 5 # 5 is because the source Q is defined as |101>_d|source_j>

xr[id_shift_regQ + id_Q_1] = coef_sqrt
xr[id_shift_regQ + id_Q_2] = coef_sqrt

# write the intial state to a file:
xw.mix.write_init_state("XW", path_root, xr, xi, norm_coef)

writing the initial state of XW


In [ ]:
2**(10+3)